# PI-LSTM v3 (Run C) — Google Colab GPU Training Run

**Steps:**
1. Mount Google Drive
2. Verify GPU (T4 / A100)
3. Auto-locate and unzip `IsotopePINN_Project.zip`
4. Install dependencies
5. Launch 6,000-epoch PI-LSTM training with `expmix` exact physics loss & v2 teacher distillation
6. Run model comparison & generate ISEF figures
7. Download `PI_LSTM_Results.zip` to your browser

In [ ]:
# ── Cell 1: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive mounted.')

In [ ]:
# ── Cell 2: Verify GPU ──
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB')
else:
    print('⚠️ NO GPU detected! Go to Runtime → Change runtime type → GPU (T4 or A100).')
    print('   Training on CPU will be extremely slow.')

In [ ]:
# ── Cell 3: Auto-locate and unzip IsotopePINN_Project.zip ──
import os, zipfile

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_DIR = os.path.join(DRIVE_ROOT, 'IsotopePINN')

found_zip = None
for path in [
    os.path.join(DRIVE_ROOT, 'IsotopePINN_Project.zip'),
    os.path.join(PROJECT_DIR, 'IsotopePINN_Project.zip'),
    '/content/IsotopePINN_Project.zip',
]:
    if os.path.exists(path):
        found_zip = path
        break

if not found_zip:
    print("Searching Drive for 'IsotopePINN_Project.zip'...")
    for root, dirs, files in os.walk(DRIVE_ROOT):
        if root[len(DRIVE_ROOT):].count(os.sep) > 2:
            continue
        if 'IsotopePINN_Project.zip' in files:
            found_zip = os.path.join(root, 'IsotopePINN_Project.zip')
            break

if not found_zip:
    raise FileNotFoundError(
        "❌ Could not find 'IsotopePINN_Project.zip' anywhere in Google Drive.\n"
        "Upload IsotopePINN_Project.zip to your Drive root and re-run this cell."
    )

print(f'Found ZIP at: {found_zip}')
os.makedirs(PROJECT_DIR, exist_ok=True)

print('Extracting project files (fixing Windows backslash paths)...')
with zipfile.ZipFile(found_zip, 'r') as zf:
    for member in zf.infolist():
        filename = member.filename.replace('\\', '/')
        dest_file = os.path.join(PROJECT_DIR, filename)
        os.makedirs(os.path.dirname(dest_file), exist_ok=True)
        if not member.is_dir() and not filename.endswith('/'):
            with zf.open(member) as src, open(dest_file, 'wb') as dst:
                dst.write(src.read())

print('✅ Project files unzipped to:', PROJECT_DIR)
!ls -la {PROJECT_DIR}

In [ ]:
# ── Cell 4: Install Dependencies ──
!pip install -q scipy>=1.11.2 matplotlib pandas joblib scikit-learn
print('✅ Dependencies installed.')

In [ ]:
# ── Cell 5: Launch PI-LSTM Training (Run C Recipe) ──
import os
os.chdir('/content/drive/MyDrive/IsotopePINN')
print(f'Working directory: {os.getcwd()}')

cmd = (
    "PILSTM_FLOAT64=1 "
    "PILSTM_EVAL_EVERY=25 "
    "PILSTM_EPOCHS=6000 "
    "PILSTM_N_TRAIN=1400 "
    "PILSTM_N_VAL=22 "
    "PILSTM_N_TEST=22 "
    "PILSTM_N_STEPS=64 "
    "PILSTM_BATCH=16 "
    "PILSTM_HIDDEN=256 "
    "PILSTM_FOURIER=8 "
    "PILSTM_TIME_FOURIER=16 "
    "PILSTM_HARD_IC=1 "
    "PILSTM_DATA_WEIGHT=35 "
    "PILSTM_PHYSICS_WEIGHT=20 "
    "PILSTM_MASS_WEIGHT=10 "
    "PILSTM_DISTILL=1 "
    "PILSTM_DISTILL_WEIGHT=5 "
    "PILSTM_OVERSHOOT_WEIGHT=20 "
    "PILSTM_CAUSAL_EPS=2.5 "
    "PILSTM_CKPT_METRIC=endpoint_ac225 "
    "python v3_pilstm/train_pi_lstm.py"
)

!{cmd}

In [ ]:
# ── Cell 6: Run Model Comparison & Generate Figures ──
import os
os.chdir('/content/drive/MyDrive/IsotopePINN')
!python v3_pilstm/analysis/compare_models.py

In [ ]:
# ── Cell 7: Download Results Zip ──
import os, zipfile
from google.colab import files

PROJECT_DIR = '/content/drive/MyDrive/IsotopePINN'
OUTPUT_ZIP = '/content/PI_LSTM_Results.zip'

result_files = [
    'v3_pilstm/weights/pi_lstm_best.pth',
    'v3_pilstm/results/model_comparison.json',
    'v3_pilstm/results/graph_manifest.json',
    'graphs/v3_v2_vs_pilstm_ac225.png',
    'graphs/v3_species_median_errors.png',
    'graphs/v3_trajectory_example.png',
    'graphs/v3_joyo_sigma_calibration.png',
    'graphs/v3_literature_anchors.png',
]

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel in result_files:
        full = os.path.join(PROJECT_DIR, rel)
        if os.path.exists(full):
            zf.write(full, rel)
            print(f'  ✅ {rel}')
        else:
            print(f'  ⚠️ {rel} (not found, skipping)')

print(f'\n📦 Results packaged: {OUTPUT_ZIP}')
print('📥 Downloading to your browser...')
files.download(OUTPUT_ZIP)